In [5]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn kagglehub

  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (8.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
Us

In [3]:
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 9.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]


In [1]:
# =========================
# IMPORTS
# =========================
import kagglehub
import pandas as pd
import os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


# =========================
# DOWNLOAD DATASET
# =========================
print("Baixando dataset do Kaggle...")
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
print("Path:", path)


# =========================
# CAMINHO DO ARQUIVO
# =========================
caminho_arquivo = os.path.join(path, "yelp_academic_dataset_review.json")


# =========================
# LEITURA SEGURA (EVITA TRAVAR)
# =========================
print("\nLendo amostra do dataset...")

df = pd.read_json(
    caminho_arquivo,
    lines=True,
    nrows=5000   # 🔥 ESSENCIAL (evita travar)
)

# Remover nulos
df = df.dropna(subset=['text'])


# =========================
# PIPELINE DE LIMPEZA
# =========================
def pipeline_limpeza(texto):
    return str(texto).lower()


reviews = df['text'].apply(pipeline_limpeza)


# =========================
# TF-IDF
# =========================
print("\nAplicando TF-IDF...")

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=3000
)

matriz_tfidf = vectorizer.fit_transform(reviews)


# =========================
# DATAFRAME
# =========================
df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)


# =========================
# TOP 10 PALAVRAS
# =========================
media = np.mean(df_tfidf, axis=0)

df_palavras = pd.DataFrame({
    'palavra': df_tfidf.columns,
    'peso_tfidf': media
})

top10 = df_palavras.sort_values(by='peso_tfidf', ascending=False).head(10)


# =========================
# RELATÓRIO
# =========================
linhas, colunas = matriz_tfidf.shape

bytes_esparsa = (
    matriz_tfidf.data.nbytes +
    matriz_tfidf.indptr.nbytes +
    matriz_tfidf.indices.nbytes
)

mb_esparsa = bytes_esparsa / (1024 * 1024)

bytes_densa = linhas * colunas * 8
mb_densa = bytes_densa / (1024 * 1024)

economia_mb = mb_densa - mb_esparsa
economia_percentual = (economia_mb / mb_densa) * 100


print("\n" + "="*50)
print("RELATÓRIO TF-IDF - YELP")
print("="*50)
print(f"Total de Reviews: {linhas}")
print(f"Vocabulário: {colunas}")
print("-" * 50)
print(f"Memória Esparsa: {mb_esparsa:.4f} MB")
print(f"Memória Densa: {mb_densa:.4f} MB")
print("-" * 50)
print(f"Economia: {economia_mb:.4f} MB ({economia_percentual:.2f}%)")
print("="*50)


# =========================
# RESULTADO FINAL
# =========================
print("\nTop 10 palavras mais relevantes:\n")
print(top10)

/workspaces/Processamento-Linguagem-Natural-PLN-/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Baixando dataset do Kaggle...
Path: /home/codespace/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4

Lendo amostra do dataset...

Aplicando TF-IDF...

RELATÓRIO TF-IDF - YELP
Total de Reviews: 5000
Vocabulário: 3000
--------------------------------------------------
Memória Esparsa: 1.9070 MB
Memória Densa: 114.4409 MB
--------------------------------------------------
Economia: 112.5339 MB (98.33%)

Top 10 palavras mais relevantes:

         palavra  peso_tfidf
great      great    0.038022
food        food    0.037372
good        good    0.035055
place      place    0.034319
service  service    0.026773
time        time    0.022708
just        just    0.020812
like        like    0.020403
really    really    0.018968
best        best    0.018468
